# 02 — Gold source, génération des notes et reconstruction

Deuxième partie du pipeline, qui utilise **MedGemma** (`google/medgemma-4b-it`) :

4. Finaliser le `gold_source` avec un filtre LLM sur les conditions → `data/gold_standard/`
5. Générer une note clinique par patient, dans 4 styles → `data/processed/notes/`
6. Reconstruire une structure depuis chaque note → `data/processed/reconstruction/`

Pré-requis : avoir exécuté `01_fhir_exploration.ipynb` (dataset filtré présent). Un GPU est fortement recommandé.

Nomenclature :
- `gold` / `gold_source` : source structurée filtrée de référence
- `raw_note_text` : sortie brute du LLM ; `note_text` : note nettoyée utilisée pour la reconstruction
- `recon_raw` : extraction brute ; `recon_clean` : reconstruction principale (scores) ; `recon_clipped` : debug (overlap exact)

In [ ]:
# Uniquement sur Google Colab : décommenter pour récupérer le dépôt et installer les dépendances
# !git clone https://github.com/<ton-compte>/bupa.git
# %cd bupa/notebooks
# !pip -q install -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path

# Rend `config` et `src` importables depuis notebooks/ (ou depuis la racine du dépôt)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import JSON, display

from config import settings

settings.ensure_dirs()
pd.set_option("display.max_colwidth", 200)
print("ROOT:", ROOT)

In [ ]:
# settings.N_PATIENTS = 40   # même valeur que dans le notebook 01

In [ ]:
# MedGemma est un modèle « gated » : il faut un token Hugging Face.
# En local : créer un fichier .env à la racine avec HF_TOKEN=hf_xxx (voir .env.example).
# Sur Colab : ajouter un secret nommé HF_TOKEN.
from src.llm import get_llm, hf_login

hf_login()
llm = get_llm()  # le modèle n'est réellement chargé qu'au premier appel
print("Model:", settings.MODEL_NAME, "| transformers/torch available:", llm.available())

## 4A) Recharger l'échantillon préfiltré (même tirage que le notebook 01)

In [ ]:
from src import fhir

df_index = fhir.build_filtered_dataset()
df_sample, patients_prefilter, df_gold_prefilter = fhir.select_sample(df_index)
display(df_gold_prefilter)

## 4B) Finaliser le `gold_source` avec le filtre LLM

Pour chaque patient, MedGemma choisit les index des conditions réellement médicales.
Résultat mis en cache dans `data/processed/filter_cache/`. En cas de doute ou d'erreur, tout est conservé.

In [ ]:
from src import gold

patients = gold.build_gold(patients_prefilter, llm=llm)
print("Gold dataset (after hybrid filter: prefilter then LLM)")
display(gold.gold_dataframe(patients))

## 5) Génération des notes non structurées

In [ ]:
from src import notes

patients = notes.assign_styles_and_save(patients)  # le style fait partie du gold
df_gold = gold.gold_dataframe(patients)
print("Note style distribution")
display(df_gold["note_style"].value_counts().to_frame("n"))

In [ ]:
raw_notes, generated_notes = notes.generate_notes(patients, llm=llm)
df_notes = notes.notes_dataframe(patients, raw_notes, generated_notes)
print("Notes dataset (note_text = cleaned note used for reconstruction)")
display(df_notes)

## 6) Reconstruction structurée depuis `note_text`

In [ ]:
from src import reconstruction

recon_raw, recon_clean, recon_clipped = reconstruction.reconstruct_all(patients, generated_notes, llm=llm)
df_recon = reconstruction.recon_dataframe(recon_raw, recon_clean, recon_clipped)
print("Reconstruction dataset (main = cleaned/no strict clipping, clipped = exact-overlap debug only)")
display(df_recon)

**Suite :** `03_benchmark_results.ipynb` (scoring, vérification, analyse des erreurs, exports, figures). Tout est relu depuis le disque, on peut donc le lancer dans une autre session.